# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @ids
print("Record Sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"  - @id: {rs.id} (name='{rs.name}')")

# For each record set, list its fields and columns by @id
print("\nRecord Set Fields and Columns (referenced by @id):\n")
for rs in dataset.metadata.record_sets:
    print(f"Record Set @id: {rs.id} ({rs.name})")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id} (name='{field.name}', dataType={field.data_type})")
            if hasattr(field, 'columns') and field.columns:
                print(f"      Columns:")
                for col in field.columns:
                    print(f"        - @id: {col.id} (name='{col.name}')")
    print()

## 3. Data Extraction
Load data from record sets into DataFrames for analysis.

Replace `record_set_ids` below with the desired Record Set `@id`s from the overview step.

In [ ]:
# Collect available record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Record sets found: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}. Shape: {df.shape}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Example: Show first few columns and records from the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns for {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filtering, normalization, removing outliers, or grouping, using field `@id`s.

*Replace `numeric_field_id` and `group_field_id` below with field @ids from previous cells corresponding to numeric and grouping fields you wish to analyze.*

In [ ]:
# Example record set and fields. Update as needed for your dataset.
record_set_id = main_rs_id  # Or pick another from record_set_ids

# Try to auto-detect a numeric field (float or int)
numeric_field_id = None
group_field_id = None
for rs in dataset.metadata.record_sets:
    if rs.id == record_set_id:
        # Find a float/int field
        for field in rs.fields:
            if getattr(field, 'data_type', None) in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number']:
                numeric_field_id = field.id
                break
        # Find a groupable (categorical/text) field
        for field in rs.fields:
            if getattr(field, 'data_type', None) in ['schema:Text', 'Text', 'schema:String', 'String']:
                group_field_id = field.id
                break
        break

print(f"Numeric field detected: {numeric_field_id}")
print(f"Group field detected: {group_field_id}")

df = dataframes.get(record_set_id)

if df is not None and numeric_field_id and numeric_field_id in df.columns:
    # Remove non-numeric or missing values
    df_num = df.copy()
    df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id], errors='coerce')
    threshold = df_num[numeric_field_id].quantile(0.75)  # Use 75th percentile as example threshold
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%): {len(filtered_df)} rows")
    
    filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group_field_id, if exists
    if group_field_id and group_field_id in df_num.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped average of {numeric_field_id} by {group_field_id} (top 5):")
        print(grouped_df.head())
else:
    print("Numeric field could not be found or is not present in the record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        top_groups = df[group_field_id].value_counts().nlargest(10).index
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
        plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded and explored the FAIR² dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. We demonstrated how to inspect available record sets and fields using their Croissant `@id` references, extract data programmatically using the `mlcroissant` library, and perform foundational EDA with visualization. For deeper analysis, consult the field descriptions and expand filtering/grouping based on your scientific questions.*